In [ ]:
def extract_spike_features(events_all, annotated_all, long):
    """
    Extracts per-patient, per-day spike features across four time windows.

    Inputs:
        events_all    — output of run_all_events (one row per event)
        annotated_all — output of run_all_events (one row per CGM reading, with baseline)
        long          — reshaped long-format CGM DataFrame

    Output:
        Wide-format DataFrame: one row per patient-day, columns per time window
    """

    TIME_WINDOWS = {
        "morning":   (6,  12),
        "afternoon": (12, 18),
        "evening":   (18, 24),
        "night":     (0,   6),
    }

    # ── Prep events (spikes only) ──────────────────────────────────────────
    spikes = events_all[events_all["direction"] == "up"].copy()
    spikes["start_time"] = pd.to_datetime(spikes["start_time"])
    spikes["end_time"]   = pd.to_datetime(spikes["end_time"])
    spikes["peak_time"]  = pd.to_datetime(spikes["peak_time"])
    spikes["Day"]        = pd.to_datetime(spikes["Day"]).dt.date
    spikes["amplitude"]  = spikes["extreme_glucose"] - spikes["baseline_at_extreme"]
    spikes["duration_min"] = (spikes["end_time"] - spikes["start_time"]).dt.total_seconds() / 60
    spikes["peak_hour"]  = spikes["peak_time"].dt.hour

    # ── Prep annotated trace ───────────────────────────────────────────────
    ann = annotated_all.copy()
    ann["datetime"] = pd.to_datetime(ann["datetime"])
    ann["date"]     = pd.to_datetime(ann["date"]).dt.date
    ann["hour"]     = ann["datetime"].dt.hour
    ann["glucose_mmol_l"] = pd.to_numeric(ann["glucose_mmol_l"], errors="coerce")
    ann["baseline"]       = pd.to_numeric(ann["baseline"],       errors="coerce")
    ann["residual"]       = ann["glucose_mmol_l"] - ann["baseline"]

    # ── Gradient: compute from annotated slope column (mmol/L per min) ────
    # slope column already exists in annotated_all from find_events
    ann["slope"] = pd.to_numeric(ann["slope"], errors="coerce")

    all_rows = []

    for pid in ann["person_id"].unique():
        ann_p    = ann[ann["person_id"] == pid]
        spikes_p = spikes[spikes["person_id"] == pid] if "person_id" in spikes.columns else spikes

        for day in ann_p["date"].unique():
            ann_day    = ann_p[ann_p["date"] == day]
            spikes_day = spikes_p[spikes_p["Day"] == day]

            row = {"person_id": pid, "day": day}

            for win_name, (h_start, h_end) in TIME_WINDOWS.items():
                # ── Slice annotated readings for this window ──
                win_ann = ann_day[
                    (ann_day["hour"] >= h_start) & (ann_day["hour"] < h_end)
                ]

                # ── Slice spikes whose peak falls in this window ──
                win_spikes = spikes_day[
                    (spikes_day["peak_hour"] >= h_start) &
                    (spikes_day["peak_hour"] <  h_end)
                ]

                p = win_name  # prefix

                # 1. Spike intensity (amplitude = peak - baseline)
                if not win_spikes.empty:
                    row[f"{p}_amp_mean"] = win_spikes["amplitude"].mean()
                    row[f"{p}_amp_max"]  = win_spikes["amplitude"].max()
                else:
                    row[f"{p}_amp_mean"] = 0.0
                    row[f"{p}_amp_max"]  = 0.0

                # 2. Gradient (slope at peak moment — from annotated trace)
                if not win_spikes.empty and not win_ann.empty:
                    peak_times = win_spikes["peak_time"]
                    slopes_at_peak = []
                    for pt in peak_times:
                        nearest_idx = (win_ann["datetime"] - pt).abs().idxmin()
                        slopes_at_peak.append(ann.loc[nearest_idx, "slope"])
                    slopes_at_peak = pd.Series(slopes_at_peak, dtype=float).dropna()
                    row[f"{p}_gradient_mean"] = slopes_at_peak.mean() if len(slopes_at_peak) else 0.0
                    row[f"{p}_gradient_max"]  = slopes_at_peak.max()  if len(slopes_at_peak) else 0.0
                else:
                    row[f"{p}_gradient_mean"] = 0.0
                    row[f"{p}_gradient_max"]  = 0.0

                # 3. Spike duration (minutes)
                if not win_spikes.empty:
                    row[f"{p}_duration_mean"] = win_spikes["duration_min"].mean()
                    row[f"{p}_duration_max"]  = win_spikes["duration_min"].max()
                else:
                    row[f"{p}_duration_mean"] = 0.0
                    row[f"{p}_duration_max"]  = 0.0

                # 4. Coefficient of variation from baseline
                #    CV = SD of residuals / mean baseline × 100
                if not win_ann.empty and win_ann["baseline"].notna().any():
                    mean_baseline = win_ann["baseline"].mean()
                    sd_residual   = win_ann["residual"].std()
                    row[f"{p}_cv_from_baseline"] = (
                        (sd_residual / mean_baseline * 100)
                        if mean_baseline and mean_baseline != 0 else np.nan
                    )
                else:
                    row[f"{p}_cv_from_baseline"] = np.nan

                # 5. Area under curve (iAUC above baseline, trapezoid)
                if not win_ann.empty and win_ann["residual"].notna().any():
                    excursion = win_ann["residual"].clip(lower=0).fillna(0).values
                    row[f"{p}_iauc"] = float(np.trapz(excursion, dx=5))  # dx=5 min intervals
                else:
                    row[f"{p}_iauc"] = 0.0

                # 6. Number of spikes
                row[f"{p}_spike_count"] = len(win_spikes)

            all_rows.append(row)

    feature_df = pd.DataFrame(all_rows)
    feature_df = feature_df.sort_values(["person_id", "day"]).reset_index(drop=True)
    return feature_df


# ── Run ────────────────────────────────────────────────────────────────────
spike_features = extract_spike_features(events_all, annotated_all, long)
print(spike_features.shape)
spike_features.to_csv("spike_features.csv", index=False)
spike_features.head()

In [ ]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def export_spike_features_xlsx(spike_features, path="spike_features.xlsx"):

    TIME_WINDOWS = ["morning", "afternoon", "evening", "night"]

    FEATURES = [
        ("amp_mean",         "Avg Amplitude\n(mmol/L)"),
        ("amp_max",          "Max Amplitude\n(mmol/L)"),
        ("gradient_mean",    "Avg Gradient\n(mmol/L/min)"),
        ("gradient_max",     "Max Gradient\n(mmol/L/min)"),
        ("duration_mean",    "Avg Duration\n(min)"),
        ("duration_max",     "Max Duration\n(min)"),
        ("cv_from_baseline", "CV from\nBaseline (%)"),
        ("iauc",             "iAUC\n(mmol/L·min)"),
        ("spike_count",      "Spike\nCount"),
    ]

    WINDOW_COLORS = {
        "morning":   {"header": "FFF176", "subheader": "FFFDE7", "data": "FFFFF8"},
        "afternoon": {"header": "A5D6A7", "subheader": "E8F5E9", "data": "F5FBF5"},
        "evening":   {"header": "90CAF9", "subheader": "E3F2FD", "data": "F3F9FF"},
        "night":     {"header": "CE93D8", "subheader": "F3E5F5", "data": "FAF3FC"},
    }

    n_features = len(FEATURES)

    wb = Workbook()
    ws = wb.active
    ws.title = "Spike Features"

    def fill(h):
        return PatternFill("solid", start_color=h, end_color=h)

    def border():
        s = Side(style="thin")
        return Border(left=s, right=s, top=s, bottom=s)

    center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left   = Alignment(horizontal="left",   vertical="center")

    # Row 1 & 2: ID headers (merged)
    for r in (1, 2):
        for ci, label in enumerate(["Patient ID", "Day"], start=1):
            cell = ws.cell(row=r, column=ci)
            if r == 1:
                cell.value = label
            cell.font      = Font(bold=True, name="Arial", size=10, color="FFFFFF")
            cell.fill      = fill("37474F")
            cell.alignment = center
            cell.border    = border()
    ws.merge_cells(start_row=1, start_column=1, end_row=2, end_column=1)
    ws.merge_cells(start_row=1, start_column=2, end_row=2, end_column=2)

    # Row 1: window group headers
    col = 3
    for win in TIME_WINDOWS:
        colors  = WINDOW_COLORS[win]
        end_col = col + n_features - 1
        ws.merge_cells(start_row=1, start_column=col, end_row=1, end_column=end_col)
        cell            = ws.cell(row=1, column=col, value=win.upper())
        cell.font       = Font(bold=True, name="Arial", size=11)
        cell.fill       = fill(colors["header"])
        cell.alignment  = center
        cell.border     = border()
        col += n_features

    # Row 2: feature sub-headers
    col = 3
    for win in TIME_WINDOWS:
        colors = WINDOW_COLORS[win]
        for _, feat_label in FEATURES:
            cell            = ws.cell(row=2, column=col, value=feat_label)
            cell.font       = Font(bold=True, name="Arial", size=9)
            cell.fill       = fill(colors["subheader"])
            cell.alignment  = center
            cell.border     = border()
            col += 1

    # Data rows
    for r_idx, (_, row) in enumerate(spike_features.iterrows(), start=3):
        shade = (r_idx % 2 == 0)

        for ci, (val, bold) in enumerate([(str(row["person_id"]), True), (str(row["day"]), False)], start=1):
            c            = ws.cell(row=r_idx, column=ci, value=val)
            c.font       = Font(name="Arial", size=9, bold=bold)
            c.fill       = fill("ECEFF1") if shade else fill("FFFFFF")
            c.alignment  = left if ci == 1 else center
            c.border     = border()

        col = 3
        for win in TIME_WINDOWS:
            colors        = WINDOW_COLORS[win]
            data_fill_hex = "E8E8E8" if shade else colors["data"]
            for feat_key, _ in FEATURES:
                val = row.get(f"{win}_{feat_key}", np.nan)
                if pd.isna(val):
                    display_val = "—"
                elif feat_key == "spike_count":
                    display_val = int(val)
                else:
                    display_val = round(float(val), 3)
                c            = ws.cell(row=r_idx, column=col, value=display_val)
                c.font       = Font(name="Arial", size=9)
                c.fill       = fill(data_fill_hex)
                c.alignment  = center
                c.border     = border()
                col += 1

    # Column widths
    ws.column_dimensions["A"].width = 14
    ws.column_dimensions["B"].width = 12
    for i in range(3, 3 + n_features * len(TIME_WINDOWS)):
        ws.column_dimensions[get_column_letter(i)].width = 13

    # Row heights
    ws.row_dimensions[1].height = 22
    ws.row_dimensions[2].height = 42
    for r in range(3, 3 + len(spike_features)):
        ws.row_dimensions[r].height = 16

    # Freeze panes
    ws.freeze_panes = "C3"

    # Thick right border between window groups
    total_rows = 2 + len(spike_features)
    col = 3
    for win in TIME_WINDOWS:
        last_col = col + n_features - 1
        for r in range(1, total_rows + 1):
            cell = ws.cell(row=r, column=last_col)
            cell.border = Border(
                left=cell.border.left, top=cell.border.top, bottom=cell.border.bottom,
                right=Side(style="medium"),
            )
        col += n_features

    # Legend sheet
    leg = wb.create_sheet("Legend")
    leg["A1"] = "Colour Legend"
    leg["A1"].font = Font(bold=True, size=12, name="Arial")
    for i, (label, color) in enumerate([
        ("Morning (06:00–12:00)",   "FFF176"),
        ("Afternoon (12:00–18:00)", "A5D6A7"),
        ("Evening (18:00–00:00)",   "90CAF9"),
        ("Night (00:00–06:00)",     "CE93D8"),
    ], start=3):
        leg.cell(row=i, column=1, value=label).font = Font(name="Arial", size=10)
        c = leg.cell(row=i, column=2, value="          ")
        c.fill = fill(color)
    leg.column_dimensions["A"].width = 28
    leg.column_dimensions["B"].width = 16

    leg["A8"] = "Feature Definitions"
    leg["A8"].font = Font(bold=True, size=11, name="Arial")
    for i, (feat, defn) in enumerate([
        ("amp_mean / amp_max",    "Avg / max glucose amplitude above baseline per spike (mmol/L)"),
        ("gradient_mean / max",   "Avg / max rate of glucose rise at spike peak (mmol/L per min)"),
        ("duration_mean / max",   "Avg / max spike duration from start to return-to-baseline (min)"),
        ("cv_from_baseline",      "SD of residuals ÷ mean baseline × 100 (%)"),
        ("iauc",                  "Incremental AUC above baseline, trapezoid rule (mmol/L·min)"),
        ("spike_count",           "Number of spikes whose peak falls in the time window"),
    ], start=10):
        leg.cell(row=i, column=1, value=feat).font = Font(bold=True, name="Arial", size=9)
        leg.cell(row=i, column=2, value=defn).font = Font(name="Arial", size=9)
    leg.column_dimensions["B"].width = 65

    wb.save(path)
    print(f"Saved → {path}")


export_spike_features_xlsx(spike_features, path="spike_features.xlsx")